# Evaluating Reinforcement Learning Models for Non-Episodic Tasks

## Overview

Unlike episodic tasks (games, navigation with clear start/end), **non-episodic (continuing) tasks** run indefinitely:
- Traffic signal control
- Server load balancing
- HVAC temperature control
- Stock trading
- Process control

**Challenge**: No natural "episode end" makes traditional evaluation metrics (episode return, success rate) inadequate.

This notebook covers:
1. Time-window based evaluation
2. Average reward rate metrics
3. Rolling/sliding window analysis
4. Stationarity and distribution-based metrics
5. Comparative evaluation methods
6. Practical implementation for traffic control

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
from typing import Dict, List, Tuple, Optional, Callable
from collections import deque
import seaborn as sns
from scipy import stats

sns.set_style('whitegrid')
np.random.seed(42)

print("✓ Imports loaded")

## 1. Average Reward Rate (Primary Metric)

For continuing tasks, we use **average reward per time step** rather than cumulative return:

$$r(\pi) = \lim_{T \to \infty} \frac{1}{T} \sum_{t=1}^{T} r_t$$

In practice, we estimate this over a fixed evaluation horizon.

In [ ]:
class ContinuousTaskEvaluator:
    """
    Comprehensive evaluator for non-episodic RL tasks.
    """
    
    def __init__(self, 
                 evaluation_horizon: int = 10000,
                 window_size: int = 1000,
                 num_runs: int = 5):
        """
        Args:
            evaluation_horizon: Total timesteps to evaluate
            window_size: Size of sliding window for metrics
            num_runs: Number of independent runs for statistical significance
        """
        self.evaluation_horizon = evaluation_horizon
        self.window_size = window_size
        self.num_runs = num_runs
        
    def evaluate_average_reward_rate(self, 
                                     policy: Callable,
                                     env,
                                     burn_in: int = 1000) -> Dict:
        """
        Evaluate average reward rate with burn-in period.
        
        Args:
            policy: Function that takes observation and returns action
            env: Environment to evaluate in
            burn_in: Initial timesteps to discard (transient behavior)
        """
        all_runs_data = []
        
        for run in range(self.num_runs):
            obs, _ = env.reset()
            rewards = []
            
            # Run for evaluation horizon
            for t in range(burn_in + self.evaluation_horizon):
                action = policy(obs)
                obs, reward, terminated, truncated, info = env.step(action)
                
                # Only collect rewards after burn-in
                if t >= burn_in:
                    rewards.append(reward)
                
                # Reset if terminated (though shouldn't happen in continuing tasks)
                if terminated or truncated:
                    obs, _ = env.reset()
            
            all_runs_data.append(rewards)
        
        # Calculate statistics across runs
        avg_rewards_per_run = [np.mean(rewards) for rewards in all_runs_data]
        
        return {
            'mean_reward_rate': np.mean(avg_rewards_per_run),
            'std_reward_rate': np.std(avg_rewards_per_run),
            'ci_95': 1.96 * np.std(avg_rewards_per_run) / np.sqrt(self.num_runs),
            'all_runs': all_runs_data,
            'per_run_means': avg_rewards_per_run
        }
    
    def evaluate_sliding_window_performance(self,
                                           policy: Callable,
                                           env) -> Dict:
        """
        Evaluate performance using sliding window approach.
        This shows how performance evolves over time.
        """
        obs, _ = env.reset()
        rewards = []
        
        # Collect rewards
        for t in range(self.evaluation_horizon):
            action = policy(obs)
            obs, reward, terminated, truncated, info = env.step(action)
            rewards.append(reward)
            
            if terminated or truncated:
                obs, _ = env.reset()
        
        # Calculate sliding window averages
        window_averages = []
        for i in range(len(rewards) - self.window_size + 1):
            window = rewards[i:i + self.window_size]
            window_averages.append(np.mean(window))
        
        return {
            'window_averages': window_averages,
            'overall_mean': np.mean(rewards),
            'window_mean': np.mean(window_averages),
            'window_std': np.std(window_averages),
            'raw_rewards': rewards
        }
    
    def evaluate_steady_state_distribution(self,
                                          policy: Callable,
                                          env,
                                          state_metric: Callable) -> Dict:
        """
        Evaluate the distribution of a state metric in steady state.
        
        Args:
            state_metric: Function that takes env and returns a metric value
                         (e.g., queue length, temperature deviation)
        """
        obs, _ = env.reset()
        metric_values = []
        
        # Burn-in period
        for _ in range(1000):
            action = policy(obs)
            obs, _, terminated, truncated, _ = env.step(action)
            if terminated or truncated:
                obs, _ = env.reset()
        
        # Collect steady-state metrics
        for _ in range(self.evaluation_horizon):
            action = policy(obs)
            obs, _, terminated, truncated, _ = env.step(action)
            
            metric_values.append(state_metric(env))
            
            if terminated or truncated:
                obs, _ = env.reset()
        
        return {
            'mean': np.mean(metric_values),
            'std': np.std(metric_values),
            'median': np.median(metric_values),
            'percentile_95': np.percentile(metric_values, 95),
            'percentile_5': np.percentile(metric_values, 5),
            'max': np.max(metric_values),
            'distribution': metric_values
        }

print("✓ ContinuousTaskEvaluator class defined")

## 2. Time-Based Performance Metrics

For continuing tasks, we care about:
- **Performance over time windows** (hourly, daily)
- **Worst-case performance** (99th percentile)
- **Stability** (variance over time)
- **Recovery time** (after disturbances)

In [ ]:
class TimeBasedMetrics:
    """Time-based performance metrics for continuing tasks."""
    
    @staticmethod
    def compute_percentile_performance(rewards: List[float], 
                                      percentiles: List[int] = [5, 25, 50, 75, 95, 99]) -> Dict:
        """Compute performance at different percentiles."""
        return {
            f'p{p}': np.percentile(rewards, p) for p in percentiles
        }
    
    @staticmethod
    def compute_stability_metrics(rewards: List[float], 
                                 window_size: int = 100) -> Dict:
        """Measure stability of performance over time."""
        # Rolling standard deviation
        rolling_std = pd.Series(rewards).rolling(window=window_size).std().dropna()
        
        # Consecutive reward differences (smoothness)
        reward_diffs = np.diff(rewards)
        
        return {
            'mean_rolling_std': np.mean(rolling_std),
            'max_rolling_std': np.max(rolling_std),
            'mean_abs_diff': np.mean(np.abs(reward_diffs)),
            'coefficient_of_variation': np.std(rewards) / np.abs(np.mean(rewards)) if np.mean(rewards) != 0 else float('inf')
        }
    
    @staticmethod
    def detect_regime_changes(rewards: List[float], 
                             window_size: int = 200) -> Dict:
        """
        Detect if the policy behavior changes over time (non-stationary).
        Uses two-sample t-test on windows.
        """
        n_windows = len(rewards) // window_size
        if n_windows < 2:
            return {'is_stationary': True, 'p_value': 1.0}
        
        # Split into windows
        windows = [rewards[i*window_size:(i+1)*window_size] 
                  for i in range(n_windows)]
        
        # Compare first and last window
        t_stat, p_value = stats.ttest_ind(windows[0], windows[-1])
        
        # Also compute trend
        window_means = [np.mean(w) for w in windows]
        trend_slope, _, _, _, _ = stats.linregress(range(len(window_means)), window_means)
        
        return {
            'is_stationary': p_value > 0.05,
            'p_value': p_value,
            't_statistic': t_stat,
            'trend_slope': trend_slope,
            'window_means': window_means
        }
    
    @staticmethod
    def compute_regret_bound(rewards: List[float], 
                            optimal_reward: float) -> Dict:
        """
        Compute cumulative regret vs optimal policy.
        
        Regret at time T: R_T = T * r* - ∑(r_t)
        where r* is optimal average reward
        """
        cumulative_rewards = np.cumsum(rewards)
        timesteps = np.arange(1, len(rewards) + 1)
        optimal_cumulative = optimal_reward * timesteps
        regret = optimal_cumulative - cumulative_rewards
        
        # Average regret per timestep
        avg_regret = regret / timesteps
        
        return {
            'total_regret': regret[-1],
            'average_regret': avg_regret[-1],
            'regret_trajectory': regret,
            'avg_regret_trajectory': avg_regret
        }

print("✓ TimeBasedMetrics class defined")

## 3. Domain-Specific Metrics

For specific continuing tasks, define domain metrics:

In [ ]:
class DomainSpecificMetrics:
    """Domain-specific metrics for different continuing task types."""
    
    @staticmethod
    def traffic_control_metrics(env) -> Dict:
        """
        Traffic signal control specific metrics.
        """
        return {
            'total_queue_length': sum(len(q) for q in env.queues),
            'max_queue_length': max(len(q) for q in env.queues),
            'avg_waiting_time': np.mean([np.mean([v.waiting_time for v in q]) if q else 0 
                                        for q in env.queues]),
            'throughput_rate': env.total_vehicles_passed / max(env.current_time, 1),
        }
    
    @staticmethod
    def temperature_control_metrics(env) -> Dict:
        """
        HVAC/temperature control metrics.
        """
        # Assuming env has temperature and target_temp attributes
        return {
            'temperature_deviation': abs(env.temperature - env.target_temp),
            'energy_usage': env.current_energy_usage,
            'comfort_score': env.comfort_metric,
        }
    
    @staticmethod
    def server_load_balancing_metrics(env) -> Dict:
        """
        Server load balancing metrics.
        """
        return {
            'avg_response_time': env.avg_response_time,
            'max_queue_length': env.max_queue,
            'dropped_requests': env.dropped_count,
            'server_utilization': env.utilization,
        }
    
    @staticmethod
    def inventory_management_metrics(env) -> Dict:
        """
        Inventory/supply chain metrics.
        """
        return {
            'stockout_rate': env.stockout_count / max(env.total_demand, 1),
            'holding_cost': env.current_holding_cost,
            'inventory_level': env.current_inventory,
            'service_level': env.fulfilled_orders / max(env.total_orders, 1),
        }

print("✓ DomainSpecificMetrics class defined")

## 4. Comparative Evaluation Framework

Compare multiple policies in continuing tasks:

In [ ]:
class ComparativeEvaluator:
    """Compare multiple policies for continuing tasks."""
    
    def __init__(self, evaluation_horizon: int = 10000):
        self.evaluation_horizon = evaluation_horizon
    
    def compare_policies(self,
                        policies: Dict[str, Callable],
                        env,
                        metrics_fn: Optional[Callable] = None,
                        n_runs: int = 5) -> pd.DataFrame:
        """
        Compare multiple policies.
        
        Args:
            policies: Dict of {policy_name: policy_function}
            env: Environment
            metrics_fn: Function that extracts metrics from env
            n_runs: Number of independent runs
        """
        results = []
        
        for policy_name, policy in policies.items():
            print(f"Evaluating {policy_name}...")
            
            for run in range(n_runs):
                obs, _ = env.reset()
                rewards = []
                metrics_over_time = [] if metrics_fn else None
                
                for t in range(self.evaluation_horizon):
                    action = policy(obs)
                    obs, reward, terminated, truncated, _ = env.step(action)
                    rewards.append(reward)
                    
                    if metrics_fn and t % 100 == 0:  # Sample metrics periodically
                        metrics_over_time.append(metrics_fn(env))
                    
                    if terminated or truncated:
                        obs, _ = env.reset()
                
                # Aggregate results for this run
                run_result = {
                    'policy': policy_name,
                    'run': run,
                    'mean_reward': np.mean(rewards),
                    'std_reward': np.std(rewards),
                    'median_reward': np.median(rewards),
                    'min_reward': np.min(rewards),
                    'max_reward': np.max(rewards),
                }
                
                # Add domain-specific metrics if provided
                if metrics_fn and metrics_over_time:
                    # Average metrics across time
                    avg_metrics = {}
                    for key in metrics_over_time[0].keys():
                        values = [m[key] for m in metrics_over_time]
                        avg_metrics[f'avg_{key}'] = np.mean(values)
                        avg_metrics[f'std_{key}'] = np.std(values)
                    run_result.update(avg_metrics)
                
                results.append(run_result)
        
        return pd.DataFrame(results)
    
    def statistical_comparison(self, 
                              results_df: pd.DataFrame,
                              metric: str = 'mean_reward') -> Dict:
        """
        Perform statistical tests to compare policies.
        """
        policies = results_df['policy'].unique()
        
        if len(policies) < 2:
            return {"error": "Need at least 2 policies to compare"}
        
        # Pairwise t-tests
        comparisons = []
        for i, policy1 in enumerate(policies):
            for policy2 in policies[i+1:]:
                data1 = results_df[results_df['policy'] == policy1][metric]
                data2 = results_df[results_df['policy'] == policy2][metric]
                
                t_stat, p_value = stats.ttest_ind(data1, data2)
                
                comparisons.append({
                    'policy1': policy1,
                    'policy2': policy2,
                    'mean1': data1.mean(),
                    'mean2': data2.mean(),
                    'difference': data1.mean() - data2.mean(),
                    't_statistic': t_stat,
                    'p_value': p_value,
                    'significant': p_value < 0.05
                })
        
        return pd.DataFrame(comparisons)

print("✓ ComparativeEvaluator class defined")

## 5. Visualization Tools for Continuing Tasks

In [ ]:
class ContinuousTaskVisualizer:
    """Visualization tools for continuing task evaluation."""
    
    @staticmethod
    def plot_sliding_window_performance(rewards: List[float],
                                       window_size: int = 100,
                                       title: str = "Sliding Window Performance"):
        """Plot sliding window average of rewards."""
        window_avg = pd.Series(rewards).rolling(window=window_size).mean()
        window_std = pd.Series(rewards).rolling(window=window_size).std()
        
        fig, ax = plt.subplots(figsize=(12, 5))
        
        # Plot raw rewards (faded)
        ax.plot(rewards, alpha=0.2, color='gray', label='Raw Rewards')
        
        # Plot sliding average
        ax.plot(window_avg, linewidth=2, color='blue', label=f'Window Avg (size={window_size})')
        
        # Plot confidence band
        ax.fill_between(range(len(rewards)),
                        window_avg - window_std,
                        window_avg + window_std,
                        alpha=0.3, color='blue')
        
        ax.set_xlabel('Timestep', fontsize=12)
        ax.set_ylabel('Reward', fontsize=12)
        ax.set_title(title, fontsize=14, fontweight='bold')
        ax.legend()
        ax.grid(True, alpha=0.3)
        
        plt.tight_layout()
        return fig
    
    @staticmethod
    def plot_reward_distribution(rewards_dict: Dict[str, List[float]]):
        """
        Plot reward distributions for multiple policies.
        """
        fig, axes = plt.subplots(1, 2, figsize=(14, 5))
        
        # Violin plot
        data_for_violin = []
        labels = []
        for name, rewards in rewards_dict.items():
            data_for_violin.append(rewards)
            labels.append(name)
        
        parts = axes[0].violinplot(data_for_violin, showmeans=True, showmedians=True)
        axes[0].set_xticks(range(1, len(labels) + 1))
        axes[0].set_xticklabels(labels, rotation=15)
        axes[0].set_ylabel('Reward', fontsize=12)
        axes[0].set_title('Reward Distribution Comparison', fontsize=13, fontweight='bold')
        axes[0].grid(True, alpha=0.3, axis='y')
        
        # Histogram/KDE
        for name, rewards in rewards_dict.items():
            axes[1].hist(rewards, bins=50, alpha=0.5, label=name, density=True)
        
        axes[1].set_xlabel('Reward', fontsize=12)
        axes[1].set_ylabel('Density', fontsize=12)
        axes[1].set_title('Reward Density', fontsize=13, fontweight='bold')
        axes[1].legend()
        axes[1].grid(True, alpha=0.3)
        
        plt.tight_layout()
        return fig
    
    @staticmethod
    def plot_performance_over_time(results_df: pd.DataFrame,
                                  metric: str = 'mean_reward'):
        """
        Plot mean performance with confidence intervals for multiple policies.
        """
        fig, ax = plt.subplots(figsize=(10, 6))
        
        policies = results_df['policy'].unique()
        
        # Group by policy and calculate statistics
        for policy in policies:
            policy_data = results_df[results_df['policy'] == policy]
            mean = policy_data[metric].mean()
            std = policy_data[metric].std()
            n = len(policy_data)
            ci = 1.96 * std / np.sqrt(n)
            
            # Bar plot with error bars
            ax.bar(policy, mean, yerr=ci, capsize=10, alpha=0.7)
        
        ax.set_ylabel(metric.replace('_', ' ').title(), fontsize=12)
        ax.set_title('Policy Comparison with 95% CI', fontsize=14, fontweight='bold')
        ax.grid(True, alpha=0.3, axis='y')
        
        plt.tight_layout()
        return fig
    
    @staticmethod
    def plot_regret_over_time(regret_trajectory: np.ndarray,
                             policy_name: str = "Policy"):
        """
        Plot cumulative regret over time.
        """
        fig, ax = plt.subplots(figsize=(10, 5))
        
        timesteps = np.arange(len(regret_trajectory))
        ax.plot(timesteps, regret_trajectory, linewidth=2)
        
        ax.set_xlabel('Timestep', fontsize=12)
        ax.set_ylabel('Cumulative Regret', fontsize=12)
        ax.set_title(f'Regret Over Time - {policy_name}', fontsize=14, fontweight='bold')
        ax.grid(True, alpha=0.3)
        
        # Add asymptotic bound reference line if applicable
        # For many algorithms, regret should be sublinear (O(√T) or O(log T))
        
        plt.tight_layout()
        return fig

print("✓ ContinuousTaskVisualizer class defined")

## 6. Practical Example: Traffic Signal Control Evaluation

In [ ]:
# Mock environment and policies for demonstration
class MockTrafficEnv:
    """Simplified mock traffic environment."""
    def __init__(self):
        self.queues = [deque() for _ in range(4)]
        self.current_time = 0
        self.total_vehicles_passed = 0
        
    def reset(self):
        self.queues = [deque() for _ in range(4)]
        self.current_time = 0
        self.total_vehicles_passed = 0
        return np.zeros(10), {}
    
    def step(self, action):
        # Simplified dynamics
        self.current_time += 1
        
        # Add random vehicles
        for i in range(4):
            if np.random.random() < 0.2:
                self.queues[i].append(self.current_time)
        
        # Process vehicles
        processed = 0
        if action == 1 and self.queues[0]:
            self.queues[0].popleft()
            processed += 1
            self.total_vehicles_passed += 1
        
        # Reward: negative of total queue length
        total_queue = sum(len(q) for q in self.queues)
        reward = -total_queue * 0.1 + processed * 1.0
        
        obs = np.random.randn(10)
        return obs, reward, False, False, {}

# Create mock policies
def random_policy(obs):
    return np.random.choice([0, 1])

def greedy_policy(obs):
    # Simple greedy: always try to process
    return 1

print("✓ Mock environment and policies created")

In [ ]:
# Example 1: Evaluate average reward rate
print("Example 1: Average Reward Rate Evaluation\n" + "="*50)

env = MockTrafficEnv()
evaluator = ContinuousTaskEvaluator(
    evaluation_horizon=5000,
    window_size=500,
    num_runs=3
)

results = evaluator.evaluate_average_reward_rate(
    policy=greedy_policy,
    env=env,
    burn_in=500
)

print(f"Mean Reward Rate: {results['mean_reward_rate']:.4f}")
print(f"Std Reward Rate: {results['std_reward_rate']:.4f}")
print(f"95% CI: ± {results['ci_95']:.4f}")
print(f"\nPer-run means: {[f'{x:.4f}' for x in results['per_run_means']]}")

In [ ]:
# Example 2: Sliding window performance
print("\nExample 2: Sliding Window Performance\n" + "="*50)

window_results = evaluator.evaluate_sliding_window_performance(
    policy=greedy_policy,
    env=env
)

print(f"Overall Mean Reward: {window_results['overall_mean']:.4f}")
print(f"Window Mean: {window_results['window_mean']:.4f}")
print(f"Window Std: {window_results['window_std']:.4f}")

# Visualize
fig = ContinuousTaskVisualizer.plot_sliding_window_performance(
    window_results['raw_rewards'],
    window_size=500,
    title="Greedy Policy Performance Over Time"
)
plt.show()

In [ ]:
# Example 3: Compare multiple policies
print("\nExample 3: Comparative Policy Evaluation\n" + "="*50)

policies = {
    'Random': random_policy,
    'Greedy': greedy_policy,
}

comparator = ComparativeEvaluator(evaluation_horizon=3000)
comparison_df = comparator.compare_policies(
    policies=policies,
    env=env,
    metrics_fn=DomainSpecificMetrics.traffic_control_metrics,
    n_runs=5
)

# Summary statistics
summary = comparison_df.groupby('policy').agg({
    'mean_reward': ['mean', 'std'],
    'median_reward': 'mean',
}).round(4)

print("\nSummary Statistics:")
print(summary)

# Statistical comparison
stat_comparison = comparator.statistical_comparison(comparison_df)
print("\nStatistical Comparison:")
print(stat_comparison)

In [ ]:
# Example 4: Stability metrics
print("\nExample 4: Stability Analysis\n" + "="*50)

stability = TimeBasedMetrics.compute_stability_metrics(
    window_results['raw_rewards'],
    window_size=200
)

print(f"Mean Rolling Std: {stability['mean_rolling_std']:.4f}")
print(f"Max Rolling Std: {stability['max_rolling_std']:.4f}")
print(f"Mean Absolute Diff: {stability['mean_abs_diff']:.4f}")
print(f"Coefficient of Variation: {stability['coefficient_of_variation']:.4f}")

# Check for regime changes
regime = TimeBasedMetrics.detect_regime_changes(
    window_results['raw_rewards'],
    window_size=1000
)

print(f"\nIs Stationary: {regime['is_stationary']}")
print(f"P-value: {regime['p_value']:.4f}")
print(f"Trend Slope: {regime['trend_slope']:.6f}")

## 7. Key Recommendations for Evaluating Non-Episodic Tasks

### Primary Metrics
1. **Average Reward Rate**: Primary metric for continuing tasks
2. **Steady-State Distribution**: Analyze what states the policy converges to
3. **Percentile Performance**: Worst-case and best-case scenarios

### Evaluation Protocol
1. **Burn-in Period**: Discard initial transient behavior (e.g., first 1000 steps)
2. **Multiple Independent Runs**: 5-10 runs for statistical significance
3. **Long Evaluation Horizon**: 10,000+ steps to capture steady-state
4. **Window-based Analysis**: Track performance stability over time

### Statistical Rigor
1. **Confidence Intervals**: Always report with error bars
2. **Significance Testing**: t-tests or Mann-Whitney for comparisons
3. **Stationarity Checks**: Verify policy behavior is stable over time
4. **Effect Size**: Not just p-values, measure practical significance

### Domain-Specific Considerations
- Define **domain metrics** beyond just reward (queue length, temperature, etc.)
- Test under **different conditions** (traffic patterns, load scenarios)
- Measure **robustness** to distribution shift
- Assess **safety constraints** (never exceed limits)

### Common Pitfalls to Avoid
1. ❌ Evaluating for too short a time (missing steady-state)
2. ❌ Not accounting for burn-in period
3. ❌ Single run evaluation (no statistical validity)
4. ❌ Ignoring variance and only reporting mean
5. ❌ Not testing on diverse scenarios
6. ❌ Comparing cumulative reward instead of average reward rate

## 8. Summary: Evaluation Checklist

When evaluating non-episodic RL:

### ✅ Metrics to Report
- [ ] Average reward rate (with CI)
- [ ] Reward distribution (mean, median, percentiles)
- [ ] Stability metrics (variance over time)
- [ ] Domain-specific KPIs
- [ ] Comparison to baselines

### ✅ Experimental Setup
- [ ] Burn-in period defined
- [ ] Evaluation horizon justified (10K+ steps)
- [ ] Multiple independent runs (5+)
- [ ] Different traffic/load patterns tested
- [ ] Random seeds controlled

### ✅ Analysis
- [ ] Sliding window visualization
- [ ] Stationarity test performed
- [ ] Statistical significance tested
- [ ] Worst-case performance analyzed
- [ ] Regret bounds computed (if optimal known)

### ✅ Reporting
- [ ] Error bars / confidence intervals shown
- [ ] Both mean and variance reported
- [ ] Distribution plots included
- [ ] Comparison with baselines
- [ ] Reproducibility information (seeds, hyperparameters)